# scib evaluation

scib compares methods on standard single-cell metrics. For imputation we care about **bio-conservation** metrics: does the imputed data preserve biological structure (cell-type clustering, neighborhood consistency)?

This starter runs Tangram's imputed AnnData through a couple of scib bio metrics as a smoke test. Later we extend to Transcriptformer + our GNN+DiT model on the same yardstick.


In [4]:
import scanpy as sc
import squidpy as sq
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import anndata
import scib

print('scib', scib.__version__)


scib 1.1.7


## Reproduce Tangram's imputed output

We re-run the Tangram pipeline (short version) here so we have `adata_imputed` + predicted cell-type labels available for scoring in this notebook's kernel.


In [5]:
adata_sp = sq.datasets.visium_hne_adata()
adata_sc = sq.datasets.sc_mouse_cortex()
print('spatial:', adata_sp.shape)
print('scRNA-seq:', adata_sc.shape)


/opt/homebrew/Caskroom/miniconda/base/envs/spatial-tx/lib/python3.10/site-packages/anndata/_io/utils.py:215: OldFormatWarning: Element '/obs/in_tissue' was written without encoding metadata.
  return func(*args, **kwargs)
/opt/homebrew/Caskroom/miniconda/base/envs/spatial-tx/lib/python3.10/site-packages/anndata/_io/utils.py:215: OldFormatWarning: Element '/obs/array_row' was written without encoding metadata.
  return func(*args, **kwargs)
/opt/homebrew/Caskroom/miniconda/base/envs/spatial-tx/lib/python3.10/site-packages/anndata/_io/utils.py:215: OldFormatWarning: Element '/obs/array_col' was written without encoding metadata.
  return func(*args, **kwargs)
/opt/homebrew/Caskroom/miniconda/base/envs/spatial-tx/lib/python3.10/site-packages/anndata/_io/utils.py:215: OldFormatWarning: Element '/obs/n_genes_by_counts' was written without encoding metadata.
  return func(*args, **kwargs)
/opt/homebrew/Caskroom/miniconda/base/envs/spatial-tx/lib/python3.10/site-packages/anndata/_io/utils.py:

spatial: (2688, 18078)
scRNA-seq: (21697, 36826)


/opt/homebrew/Caskroom/miniconda/base/envs/spatial-tx/lib/python3.10/site-packages/anndata/_io/utils.py:215: OldFormatWarning: Element '/raw/var/mt' was written without encoding metadata.
  return func(*args, **kwargs)
/opt/homebrew/Caskroom/miniconda/base/envs/spatial-tx/lib/python3.10/site-packages/anndata/_io/utils.py:215: OldFormatWarning: Element '/raw/var/n_cells_by_counts' was written without encoding metadata.
  return func(*args, **kwargs)
/opt/homebrew/Caskroom/miniconda/base/envs/spatial-tx/lib/python3.10/site-packages/anndata/_io/utils.py:215: OldFormatWarning: Element '/raw/var/mean_counts' was written without encoding metadata.
  return func(*args, **kwargs)
/opt/homebrew/Caskroom/miniconda/base/envs/spatial-tx/lib/python3.10/site-packages/anndata/_io/utils.py:215: OldFormatWarning: Element '/raw/var/log1p_mean_counts' was written without encoding metadata.
  return func(*args, **kwargs)
/opt/homebrew/Caskroom/miniconda/base/envs/spatial-tx/lib/python3.10/site-packages/an

In [6]:
# raw counts -> normalize (same as tangram_exploration.ipynb)
adata_sc_raw = adata_sc.raw.to_adata()
adata_sp_raw = adata_sp.raw.to_adata()

shared_genes = list(set(adata_sc_raw.var_names) & set(adata_sp_raw.var_names))
adata_sc_shared = adata_sc_raw[:, shared_genes].copy()
adata_sp_shared = adata_sp_raw[:, shared_genes].copy()
adata_sc_shared.obs = adata_sc.obs.copy()

sc.pp.normalize_total(adata_sc_shared, target_sum=1e4); sc.pp.log1p(adata_sc_shared)
sc.pp.normalize_total(adata_sp_shared, target_sum=1e4); sc.pp.log1p(adata_sp_shared)

cell_type_col = 'cell_subclass'
sc.tl.rank_genes_groups(adata_sc_shared, groupby=cell_type_col, method='wilcoxon', n_genes=100)
training_genes = sc.get.rank_genes_groups_df(adata_sc_shared, group=None)['names'].unique().tolist()
training_genes = [g for g in training_genes if g in adata_sp_shared.var_names]
print(f'training genes: {len(training_genes)} | cell types: {adata_sc_shared.obs[cell_type_col].nunique()}')


training genes: 1628 | cell types: 23


/opt/homebrew/Caskroom/miniconda/base/envs/spatial-tx/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:458: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "names"] = self.var_names[global_indices]
/opt/homebrew/Caskroom/miniconda/base/envs/spatial-tx/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:460: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "scores"] = scores[global_indices]
/opt/homebrew/Caskroom/miniconda/base/envs/spatial-tx/lib/python3.10/site

In [7]:
# short Tangram training (200 iters is enough for a quick eval smoke test)
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

def to_dense(X):
    return X.toarray() if hasattr(X, 'toarray') else np.asarray(X)

S_train = torch.tensor(to_dense(adata_sc_shared[:, training_genes].X), dtype=torch.float32, device=device)
T_train = torch.tensor(to_dense(adata_sp_shared[:, training_genes].X), dtype=torch.float32, device=device)
n_cells, n_spots = S_train.shape[0], T_train.shape[0]

M_raw = torch.nn.Parameter(torch.zeros(n_cells, n_spots, device=device))
opt = torch.optim.Adam([M_raw], lr=0.1)

for i in range(200):
    opt.zero_grad()
    M = F.softmax(M_raw, dim=1)
    S_pred = M.T @ S_train
    cos = F.cosine_similarity(S_pred, T_train, dim=1).mean()
    density = M.sum(dim=0); density = density / density.sum()
    kl = (density * (torch.log(density + 1e-8) - torch.log(torch.ones(n_spots, device=device) / n_spots))).sum()
    loss = -cos + kl
    loss.backward(); opt.step()
    if i % 50 == 0:
        print(f'iter {i:3d} loss {loss.item():.4f}')
print('done')


iter   0 loss -0.7784
iter  50 loss -0.7835
iter 100 loss -0.7883
iter 150 loss -0.7909
done


In [8]:
# Build adata_imputed and predicted cell-type labels
with torch.no_grad():
    M_np = F.softmax(M_raw, dim=1).cpu().numpy()

S_all = torch.tensor(to_dense(adata_sc_shared.X), dtype=torch.float32, device=device)
imputed = (F.softmax(M_raw, dim=1).T @ S_all).detach().cpu().numpy()

adata_imputed = anndata.AnnData(X=imputed)
adata_imputed.obs_names = adata_sp.obs_names
adata_imputed.var_names = adata_sc_shared.var_names
adata_imputed.obsm['spatial'] = adata_sp.obsm['spatial']

cell_types = adata_sc_shared.obs[cell_type_col].values
unique_types = np.unique(cell_types)
ct_probs = np.zeros((n_spots, len(unique_types)))
for j, ct in enumerate(unique_types):
    ct_probs[:, j] = M_np[cell_types == ct].sum(axis=0)
predicted = np.array([unique_types[i] for i in ct_probs.argmax(axis=1)])
adata_imputed.obs['predicted_cell_type'] = pd.Categorical(predicted)
print('adata_imputed:', adata_imputed.shape, '| unique predicted types:', len(np.unique(predicted)))


adata_imputed: (2688, 16311) | unique predicted types: 19


## scib bio-conservation metrics

We compute:
- neighborhood graph on imputed expression
- leiden clustering
- **ARI** and **NMI** between leiden clusters and Tangram's predicted cell types (consistency check)
- **silhouette** on cell-type labels (how well types separate in PCA space)

For a real comparison you'd score every method (Tangram, Transcriptformer, GNN+DiT) with the same metrics against the SAME label set. This is a pipeline smoke test.


In [9]:
sc.pp.pca(adata_imputed, n_comps=50)
sc.pp.neighbors(adata_imputed, n_neighbors=15, use_rep='X_pca')
sc.tl.leiden(adata_imputed, resolution=1.0)
print('leiden clusters:', adata_imputed.obs['leiden'].nunique())


/var/folders/fz/yyjmfcln0qg_xl49_bp8ycvr0000gn/T/ipykernel_53676/3239369656.py:3: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(adata_imputed, resolution=1.0)


leiden clusters: 21


In [10]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

labels_true = adata_imputed.obs['predicted_cell_type'].astype(str).values
labels_pred = adata_imputed.obs['leiden'].astype(str).values

ari = adjusted_rand_score(labels_true, labels_pred)
nmi = normalized_mutual_info_score(labels_true, labels_pred)
print(f'ARI: {ari:.4f}')
print(f'NMI: {nmi:.4f}')


ARI: 0.2021
NMI: 0.5299


In [11]:
# scib silhouette on cell-type labels (bio-conservation)
sil = scib.metrics.silhouette(adata_imputed, label_key='predicted_cell_type', embed='X_pca')
print(f'silhouette (cell-type): {sil:.4f}')


silhouette (cell-type): 0.5247


/opt/homebrew/Caskroom/miniconda/base/envs/spatial-tx/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/homebrew/Caskroom/miniconda/base/envs/spatial-tx/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/homebrew/Caskroom/miniconda/base/envs/spatial-tx/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


## Interpreting the numbers

- **ARI / NMI**: how consistent leiden clusters are with Tangram's predicted labels. Since the labels come from Tangram itself, high values here only mean Tangram is internally consistent — not that it is biologically correct.
- **silhouette**: how well cell-type clusters are separated in PCA space. Range [-1, 1], higher = better separation.

## Next steps

1. Get true labels for Visium spots (transfer from scRNA-seq via `sc.tl.ingest` or scANVI). Use those as `label_key` for real biology scoring.
2. Once Transcriptformer + our GNN+DiT model produce outputs, score them with the same metrics on the same label set to compare methods fairly.
3. Add batch-correction metrics only if we mix multiple datasets/donors.
